In [1]:
import numpy as np
import cv2 as cv
import time
import matplotlib.pyplot as plt
from tkinter import *
import tkinter as tk
from PIL import Image, ImageTk
import random   

window = tk.Tk()

# MAIN WINDOW SETTINGS
imgs = tk.PhotoImage(file="C:\\Users\\HP\\Documents\\Object Detection\\Project\\Object Detection In Real Time\\finalize.png")

window.title("OBJECT DETECTION")
window.geometry('1100x650')
window.configure(background='black')

LabelImg = Label(image=imgs)
LabelImg.pack()

message = tk.Label(window, text="OBJECT DETECTION", bg="light blue", fg="black", 
                   width=48, height=2, font=('times', 30, 'italic bold'))
message.place(x=50, y=10)


# --------------------------------------------------------------------
# SAFE WINDOW CHECK (PREVENTS CRASH)
# --------------------------------------------------------------------
def window_alive():
    try:
        return window.winfo_exists() == 1
    except:
        return False


# --------------------------------------------------------------------
# EXIT FUNCTION
# --------------------------------------------------------------------
def exit():
    try:
        cap.release()
    except:
        pass
    
    if window_alive():
        window.destroy()


# --------------------------------------------------------------------
# RUN ON IMAGE
# --------------------------------------------------------------------
def runToTheImage():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    images = [
        "person.jpg", "cats.jpg", "dog.jpg", "electro.png",
        "reading.jpg", "sofa.jpg", "street.jpg", "bus.jpg","Hand.jpg","Finger.jpg","pen.jpg","laptop.jpg","train.jpg","airplane.jpg","motorcycle.jpg","book.jpg","watch.jpg","spects.jpg,
    ]

    number = random.randint(0, len(images)-1)
    img = cv.imread(images[number])

    rows, cols = img.shape[:2]

    blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                (127.5, 127.5, 127.5), swapRB=True, crop=False)
    tf_net.setInput(blob)
    out = tf_net.forward()

    confidence = 0.5

    for detection in out[0, 0, :, :]:
        score = float(detection[2])
        if score > confidence:
            label = int(detection[1]) - 1
            left = int(detection[3] * cols)
            top = int(detection[4] * rows)
            right = int(detection[5] * cols)
            bottom = int(detection[6] * rows)

            text = f"{labels[label]} - {round(score*100, 2)}%"
            cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                       0.5, label_colors[label], 2)
            cv.rectangle(img, (left, top), (right, bottom),
                         label_colors[label], 3)

    if not window_alive():
        return

    imageFrame = Frame(window, bg="grey", width=700, height=600)
    rst = tk.Label(imageFrame, background="snow", fg="black", font=("", 15))

    out_img = cv.resize(img, (600, 500))
    rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
    imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

    rst.config(image=imgTk)
    rst.image = imgTk
    rst.place(x=50, y=40)

    imageFrame.place(x=250, y=120)

    def clear():
        imageFrame.destroy()

    Button(window, text="Clear", command=clear, fg="black", bg="lawn green",
           padx=10, pady=10, width=10, height=2, font=('times', 15, 'bold')).place(x=1000, y=350)


# --------------------------------------------------------------------
# RUN ON VIDEO
# --------------------------------------------------------------------
def runToTheVideo():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    cap = cv.VideoCapture("video1.mp4")
    pause = True

    while pause:

        if not window_alive():
            cap.release()
            return

        succ, img = cap.read()
        if not succ:
            break

        img = cv.flip(img, 1)

        rows, cols = img.shape[:2]

        blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                    (127.5, 127.5, 127.5), swapRB=True, crop=False)

        tf_net.setInput(blob)
        out = tf_net.forward()

        confidence = 0.5

        for detection in out[0, 0, :, :]:
            score = float(detection[2])
            if score > confidence:
                label = int(detection[1]) - 1

                left = int(detection[3] * cols)
                top = int(detection[4] * rows)
                right = int(detection[5] * cols)
                bottom = int(detection[6] * rows)

                text = f"{labels[label]} - {round(score*100, 2)}%"
                cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                           0.6, label_colors[label], 2)
                cv.rectangle(img, (left, top), (right, bottom),
                             label_colors[label], 2)

        if not window_alive():
            cap.release()
            return

        imageFrame = Frame(window, bg="sky blue", width=700, height=600)
        rst = tk.Label(imageFrame, background="snow", fg="black")

        out_img = cv.resize(img, (600, 500))
        rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
        imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

        rst.config(image=imgTk)
        rst.image = imgTk
        rst.place(x=50, y=40)
        imageFrame.place(x=250, y=120)

        try:
            window.update()
        except:
            break

    cap.release()
    cv.destroyAllWindows()


# --------------------------------------------------------------------
# RUN LIVE CAMERA
# --------------------------------------------------------------------
def runLive():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    cap = cv.VideoCapture(0)
    pause = True

    while pause:

        if not window_alive():
            cap.release()
            return

        succ, img = cap.read()
        if not succ:
            break

        img = cv.flip(img, 1)
        rows, cols = img.shape[:2]

        blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                    (127.5, 127.5, 127.5), swapRB=True, crop=False)
        tf_net.setInput(blob)
        out = tf_net.forward()

        confidence = 0.7

        for detection in out[0, 0, :, :]:
            score = float(detection[2])
            if score > confidence:
                label = int(detection[1]) - 1

                left = int(detection[3] * cols)
                top = int(detection[4] * rows)
                right = int(detection[5] * cols)
                bottom = int(detection[6] * rows)

                text = f"{labels[label]} - {round(score*100, 2)}%"
                cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                           0.5, label_colors[label], 2)
                cv.rectangle(img, (left, top), (right, bottom),
                             label_colors[label], 3)

        if not window_alive():
            cap.release()
            return

        imageFrame = Frame(window, bg="sky blue", width=700, height=600)
        rst = tk.Label(imageFrame, background="snow", fg="black")

        out_img = cv.resize(img, (600, 500))
        rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
        imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

        rst.config(image=imgTk)
        rst.image = imgTk
        rst.place(x=50, y=40)
        imageFrame.place(x=250, y=120)

        try:
            window.update()
        except:
            break

    cap.release()
    cv.destroyAllWindows()


# --------------------------------------------------------------------
# QUIT WINDOW HANDLER
# --------------------------------------------------------------------
def on_closing():
    from tkinter import messagebox
    if messagebox.askokcancel("Quit", "Do you want to quit?"):
        try:
            cv.destroyAllWindows()
        except:
            pass
        window.destroy()

window.protocol("WM_DELETE_WINDOW", on_closing)


# --------------------------------------------------------------------
# GUI BUTTONS
# --------------------------------------------------------------------
tk.Button(window, text="video", fg="white", bg="lawn green", command=runToTheVideo,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=350)

tk.Button(window, text="Image", fg="white", bg="lawn green", command=runToTheImage,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=450)

tk.Button(window, text="Live", fg="white", bg="lawn green", command=runLive,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=250)

tk.Button(window, text="Quit", fg="white", bg="red", command=on_closing,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=1000, y=250)

window.mainloop()


SyntaxError: EOL while scanning string literal (288120442.py, line 72)

In [2]:
import numpy as np
import cv2 as cv
import time
import matplotlib.pyplot as plt
from tkinter import *
import tkinter as tk
from PIL import Image, ImageTk
import random   

window = tk.Tk()

# MAIN WINDOW SETTINGS
imgs = tk.PhotoImage(file="C:\\Users\\HP\\Documents\\Object Detection\\Project\\Object Detection In Real Time\\finalize.png")

window.title("OBJECT DETECTION")
window.geometry('1100x650')
window.configure(background='black')

LabelImg = Label(image=imgs)
LabelImg.pack()

message = tk.Label(window, text="OBJECT DETECTION", bg="light blue", fg="black", 
                   width=48, height=2, font=('times', 30, 'italic bold'))
message.place(x=50, y=10)


# --------------------------------------------------------------------
# SAFE WINDOW CHECK (PREVENTS CRASH)
# --------------------------------------------------------------------
def window_alive():
    try:
        return window.winfo_exists() == 1
    except:
        return False


# --------------------------------------------------------------------
# EXIT FUNCTION
# --------------------------------------------------------------------
def exit():
    try:
        cap.release()
    except:
        pass
    
    if window_alive():
        window.destroy()


# --------------------------------------------------------------------
# RUN ON IMAGE
# --------------------------------------------------------------------
def runToTheImage():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    images = [
        "person.jpg", "cats.jpg", "dog.jpg", "electro.png",
        "reading.jpg", "sofa.jpg", "street.jpg", "bus.jpg","Hand.jpg","Finger.jpg","pen.jpg"
    ]

    number = random.randint(0, len(images)-1)
    img = cv.imread(images[number])

    rows, cols = img.shape[:2]

    blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                (127.5, 127.5, 127.5), swapRB=True, crop=False)
    tf_net.setInput(blob)
    out = tf_net.forward()

    confidence = 0.5

    for detection in out[0, 0, :, :]:
        score = float(detection[2])
        if score > confidence:
            label = int(detection[1]) - 1
            left = int(detection[3] * cols)
            top = int(detection[4] * rows)
            right = int(detection[5] * cols)
            bottom = int(detection[6] * rows)

            text = f"{labels[label]} - {round(score*100, 2)}%"
            cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                       0.5, label_colors[label], 2)
            cv.rectangle(img, (left, top), (right, bottom),
                         label_colors[label], 3)

    if not window_alive():
        return

    imageFrame = Frame(window, bg="grey", width=700, height=600)
    rst = tk.Label(imageFrame, background="snow", fg="black", font=("", 15))

    out_img = cv.resize(img, (600, 500))
    rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
    imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

    rst.config(image=imgTk)
    rst.image = imgTk
    rst.place(x=50, y=40)

    imageFrame.place(x=250, y=120)

    def clear():
        imageFrame.destroy()

    Button(window, text="Clear", command=clear, fg="black", bg="lawn green",
           padx=10, pady=10, width=10, height=2, font=('times', 15, 'bold')).place(x=1000, y=350)


# --------------------------------------------------------------------
# RUN ON VIDEO
# --------------------------------------------------------------------
def runToTheVideo():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    cap = cv.VideoCapture("video1.mp4")
    pause = True

    while pause:

        if not window_alive():
            cap.release()
            return

        succ, img = cap.read()
        if not succ:
            break

        img = cv.flip(img, 1)

        rows, cols = img.shape[:2]

        blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                    (127.5, 127.5, 127.5), swapRB=True, crop=False)

        tf_net.setInput(blob)
        out = tf_net.forward()

        confidence = 0.5

        for detection in out[0, 0, :, :]:
            score = float(detection[2])
            if score > confidence:
                label = int(detection[1]) - 1

                left = int(detection[3] * cols)
                top = int(detection[4] * rows)
                right = int(detection[5] * cols)
                bottom = int(detection[6] * rows)

                text = f"{labels[label]} - {round(score*100, 2)}%"
                cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                           0.6, label_colors[label], 2)
                cv.rectangle(img, (left, top), (right, bottom),
                             label_colors[label], 2)

        if not window_alive():
            cap.release()
            return

        imageFrame = Frame(window, bg="sky blue", width=700, height=600)
        rst = tk.Label(imageFrame, background="snow", fg="black")

        out_img = cv.resize(img, (600, 500))
        rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
        imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

        rst.config(image=imgTk)
        rst.image = imgTk
        rst.place(x=50, y=40)
        imageFrame.place(x=250, y=120)

        try:
            window.update()
        except:
            break

    cap.release()
    cv.destroyAllWindows()


# --------------------------------------------------------------------
# RUN LIVE CAMERA
# --------------------------------------------------------------------
def runLive():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    cap = cv.VideoCapture(0)
    pause = True

    while pause:

        if not window_alive():
            cap.release()
            return

        succ, img = cap.read()
        if not succ:
            break

        img = cv.flip(img, 1)
        rows, cols = img.shape[:2]

        blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                    (127.5, 127.5, 127.5), swapRB=True, crop=False)
        tf_net.setInput(blob)
        out = tf_net.forward()

        confidence = 0.7

        for detection in out[0, 0, :, :]:
            score = float(detection[2])
            if score > confidence:
                label = int(detection[1]) - 1

                left = int(detection[3] * cols)
                top = int(detection[4] * rows)
                right = int(detection[5] * cols)
                bottom = int(detection[6] * rows)

                text = f"{labels[label]} - {round(score*100, 2)}%"
                cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                           0.5, label_colors[label], 2)
                cv.rectangle(img, (left, top), (right, bottom),
                             label_colors[label], 3)

        if not window_alive():
            cap.release()
            return

        imageFrame = Frame(window, bg="sky blue", width=700, height=600)
        rst = tk.Label(imageFrame, background="snow", fg="black")

        out_img = cv.resize(img, (600, 500))
        rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
        imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

        rst.config(image=imgTk)
        rst.image = imgTk
        rst.place(x=50, y=40)
        imageFrame.place(x=250, y=120)

        try:
            window.update()
        except:
            break

    cap.release()
    cv.destroyAllWindows()


# --------------------------------------------------------------------
# QUIT WINDOW HANDLER
# --------------------------------------------------------------------
def on_closing():
    from tkinter import messagebox
    if messagebox.askokcancel("Quit", "Do you want to quit?"):
        try:
            cv.destroyAllWindows()
        except:
            pass
        window.destroy()

window.protocol("WM_DELETE_WINDOW", on_closing)


# --------------------------------------------------------------------
# GUI BUTTONS
# --------------------------------------------------------------------
tk.Button(window, text="video", fg="white", bg="lawn green", command=runToTheVideo,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=350)

tk.Button(window, text="Image", fg="white", bg="lawn green", command=runToTheImage,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=450)

tk.Button(window, text="Live", fg="white", bg="lawn green", command=runLive,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=250)

tk.Button(window, text="Quit", fg="white", bg="red", command=on_closing,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=1000, y=250)

window.mainloop()


In [ ]:
import threading
import time
import random
import numpy as np
import cv2 as cv
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import messagebox
from ultralytics import YOLO

# ------------------------------
# CONFIG
# ------------------------------
MODEL_NAME = "yolov8n.pt"   # small & fast; change to yolov8s.pt or yolov8m.pt as desired
CONFIDENCE = 0.25           # detection confidence threshold
IMG_DISPLAY_SIZE = (600, 500)  # width, height for display in GUI

# ------------------------------
# GLOBALS
# ------------------------------
window = tk.Tk()
window.title("OBJECT DETECTION (YOLOv8)")
window.geometry('1100x650')
window.configure(background='black')

# load a background image if you want (optional); comment if unavailable
try:
    bg_img = tk.PhotoImage(file="C:\\Users\\HP\\Documents\\Object Detection\\Project\\Object Detection In Real Time\\finalize.png")
    LabelImg = tk.Label(image=bg_img)
    LabelImg.pack()
except Exception:
    # ignore if image not found
    pass

message = tk.Label(window, text="OBJECT DETECTION (YOLOv8)", bg="light blue", fg="black",
                   width=48, height=2, font=('times', 30, 'italic bold'))
message.place(x=50, y=10)

# container where detection images are shown
image_frame = tk.Frame(window, bg="grey", width=700, height=600)
image_frame.place(x=250, y=120)
display_label = tk.Label(image_frame, background="snow")
display_label.place(x=50, y=40)

# control flags
stop_event = threading.Event()   # used to signal threads to stop
model = None                     # will hold the YOLO model
colors = None                    # color palette for classes

# ------------------------------
# UTILITIES
# ------------------------------
def load_model():
    global model, colors
    if model is None:
        # load YOLOv8 model (downloads weights if necessary)
        model = YOLO(MODEL_NAME)
        # pick colors for up to 100 classes
        colors = np.random.randint(0, 255, size=(len(model.names), 3), dtype=int)

def draw_boxes(frame, boxes_xyxy, scores, class_ids, names):
    """
    Draw bounding boxes on frame (in-place). boxes_xyxy: N x 4 numpy array
    """
    for i, (xyxy, score, cid) in enumerate(zip(boxes_xyxy, scores, class_ids)):
        x1, y1, x2, y2 = map(int, xyxy)
        label = names[int(cid)]
        conf_text = f"{label} {float(score):.2f}"
        col = tuple(map(int, colors[int(cid) % len(colors)]))
        cv.rectangle(frame, (x1, y1), (x2, y2), col, thickness=2)
        # put a filled rectangle behind text for readability
        (w, h), _ = cv.getTextSize(conf_text, cv.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv.rectangle(frame, (x1, y1 - h - 6), (x1 + w + 6, y1), col, -1)
        cv.putText(frame, conf_text, (x1 + 3, y1 - 4), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1, cv.LINE_AA)

def frame_to_tk(frame, size=IMG_DISPLAY_SIZE):
    """Convert BGR frame (OpenCV) to ImageTk.PhotoImage with resizing"""
    out_img = cv.resize(frame, size)
    rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
    im_pil = Image.fromarray(rgb)
    return ImageTk.PhotoImage(im_pil)

def safe_update_image(tk_img):
    """Update the display_label in main thread"""
    display_label.config(image=tk_img)
    display_label.image = tk_img

# ------------------------------
# RUN ON IMAGE
# ------------------------------
def run_image_thread():
    def worker():
        load_model()
        # sample images list (update paths if needed)
        images = [
            "person.jpg", "cats.jpg", "dog.jpg", "electro.png",
            "reading.jpg", "sofa.jpg", "street.jpg", "bus.jpg", "Hand.jpg", "Finger.jpg", "pen.jpg", "redmi.jpg"
        ]
        number = random.randint(0, len(images)-1)
        img_path = images[number]
        frame = cv.imread(img_path)
        if frame is None:
            messagebox.showerror("Error", f"Could not read image: {img_path}")
            return

        # run inference
        results = model.predict(frame, conf=CONFIDENCE, imgsz=640)  # list of Results
        r = results[0]
        if hasattr(r, 'boxes') and len(r.boxes) > 0:
            boxes = r.boxes.xyxy.cpu().numpy()      # N x 4
            scores = r.boxes.conf.cpu().numpy()     # N
            class_ids = r.boxes.cls.cpu().numpy()   # N
            draw_boxes(frame, boxes, scores, class_ids, model.names)
        tk_img = frame_to_tk(frame)
        window.after(0, safe_update_image, tk_img)

    stop_event.clear()
    t = threading.Thread(target=worker, daemon=True)
    t.start()

# ------------------------------
# RUN ON VIDEO (file)
# ------------------------------
def run_video_thread():
    def worker():
        load_model()
        cap = cv.VideoCapture("video1.mp4")
        if not cap.isOpened():
            messagebox.showerror("Error", "Could not open video1.mp4")
            return
        stop_event.clear()
        while not stop_event.is_set():
            ret, frame = cap.read()
            if not ret:
                break
            # optional flip
            frame = cv.flip(frame, 1)
            results = model.predict(frame, conf=CONFIDENCE, imgsz=640)
            r = results[0]
            if hasattr(r, 'boxes') and len(r.boxes) > 0:
                boxes = r.boxes.xyxy.cpu().numpy()
                scores = r.boxes.conf.cpu().numpy()
                class_ids = r.boxes.cls.cpu().numpy()
                draw_boxes(frame, boxes, scores, class_ids, model.names)
            tk_img = frame_to_tk(frame)
            # update GUI in main thread
            window.after(0, safe_update_image, tk_img)
            # small sleep to yield (control playback speed)
            time.sleep(0.02)
        cap.release()

    # stop any current worker
    stop_event.set()
    time.sleep(0.05)
    stop_event.clear()
    t = threading.Thread(target=worker, daemon=True)
    t.start()

# ------------------------------
# RUN LIVE CAMERA
# ------------------------------
def run_live_thread():
    def worker():
        load_model()
        cap = cv.VideoCapture(0)
        if not cap.isOpened():
            messagebox.showerror("Error", "Could not open webcam")
            return
        stop_event.clear()
        while not stop_event.is_set():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv.flip(frame, 1)
            results = model.predict(frame, conf=CONFIDENCE, imgsz=640)
            r = results[0]
            if hasattr(r, 'boxes') and len(r.boxes) > 0:
                boxes = r.boxes.xyxy.cpu().numpy()
                scores = r.boxes.conf.cpu().numpy()
                class_ids = r.boxes.cls.cpu().numpy()
                draw_boxes(frame, boxes, scores, class_ids, model.names)
            tk_img = frame_to_tk(frame)
            window.after(0, safe_update_image, tk_img)
            # small sleep to reduce CPU
            time.sleep(0.02)
        cap.release()

    # stop any current worker
    stop_event.set()
    time.sleep(0.05)
    stop_event.clear()
    t = threading.Thread(target=worker, daemon=True)
    t.start()

# ------------------------------
# CONTROLS
# ------------------------------
def stop_detection():
    stop_event.set()
    # Optionally clear image display
    display_label.config(image="")
    display_label.image = None

def on_closing():
    if messagebox.askokcancel("Quit", "Do you want to quit?"):
        stop_event.set()
        try:
            cv.destroyAllWindows()
        except:
            pass
        window.destroy()

window.protocol("WM_DELETE_WINDOW", on_closing)

# Buttons
tk.Button(window, text="Live", fg="white", bg="lawn green", command=run_live_thread,
          width=10, height=2, padx=10, pady=10, relief=tk.SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=250)

tk.Button(window, text="Video", fg="white", bg="lawn green", command=run_video_thread,
          width=10, height=2, padx=10, pady=10, relief=tk.SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=350)

tk.Button(window, text="Image", fg="white", bg="lawn green", command=run_image_thread,
          width=10, height=2, padx=10, pady=10, relief=tk.SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=450)

tk.Button(window, text="Stop", fg="black", bg="yellow", command=stop_detection,
          width=10, height=2, padx=10, pady=10, relief=tk.RAISED,
          font=('times', 15, 'bold')).place(x=1000, y=350)

tk.Button(window, text="Quit", fg="white", bg="red", command=on_closing,
          width=10, height=2, padx=10, pady=10, relief=tk.SUNKEN,
          font=('times', 15, 'bold')).place(x=1000, y=250)

window.mainloop()




In [ ]:
from ultralytics import YOLO
print("YOLOv8 Loaded Successfully!")

